# Pricing from JSON

End-to-end book MC pricing via `pricing_engine`.

**JSON fields & products:** [`data/pricing_json_schema.yaml`](../../data/pricing_json_schema.yaml)



| File | Role |
|------|------|
| `data/market_snapshot*.json` | one or more market files (looped in order); same book for all |
| `data/options_book.json` | option specs (`options` list); generate with [`generate_options_book.ipynb`](generate_options_book.ipynb) |

## How to use this notebook

1. Build the pybind module (above).
2. Set parameters in the first code cell (`MAX_HORIZON_YEARS`, `MC_SAMPLES`, …).
3. Optionally set `GENERATE_TEST_SNAPSHOTS = N` to write N shifted copies of the base market (asof +1 business day each).
4. Loop: for every `data/market_snapshot*.json` → calibrate, simulate (horizon from that `asof`), `price_all`.
5. Inspect the summary cell: per-file errors and timing.



## Parameters (first code cell)

| Variable | Role |
|------|------|
| `MARKET_DATA_DIR`, `OPTIONS_JSON` | Input paths |
| `GENERATE_TEST_SNAPSHOTS` | `0` = use existing files only; `N` = write `market_snapshot_shift_00.json` … from base (`asof` +1 business day per step) |
| `MAX_HORIZON_YEARS` | Cap on MC horizon per market: `min(max book expiry, asof + N calendar years)` |
| `MC_SAMPLES` | Paths per simulation |
| `DYNAMICS` | `"lsv"` (default) or `"lv"` — Bergomi params come from the market JSON (`bergomi_k/nu/rho`) |
| `MC_THREADS` | OpenMP workers for MC simulate + `price_all` (written to `OMP_NUM_THREADS` before import; C++ default 6 if unset) |



## Constraints

- Dates: ISO `YYYY-MM-DD`; calendar is TARGET (use business days). All option dates must be strictly after `market["asof"]`. Alternatively set `expiry_years` (e.g. `4.0`): resolved as `asof + round(365·T)` days, adjusted to the first TARGET business day (Following). Mutually exclusive with `expiry`.
- MC grid: `expiry` and every `observation_dates` entry must fall on a business day included in the simulated path (daily TARGET schedule to the sim horizon). Weekends/holidays → `not on save path`.
- Monitoring: path-dependent products default to monthly observations when `observation_dates` is omitted; set `observation_frequency: "daily"` to monitor on every simulated business day (mutually exclusive with `observation_dates`).
- Horizon: simulation ends at `min(latest book date, asof + MAX_HORIZON_YEARS)` (also considers `observation_dates`). Options beyond that horizon cannot be priced.
- `id`: must be unique in the book (the notebook keys results by `id`).
- `product`: one of `european`, `digital`, `digital_accrual`, `asian`, `barrier`, `lookback`, `autocall`, see YAML for per-product fields.
- Order: call `simulate_paths` before `price_all`; both LV and LSV only need `calibration()`.
- Strikes / barriers: prefer `*_fraction_of_spot`; absolute `strike` / `barrier_*` also supported.

In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

# OpenMP workers for simulate_paths + price_all (must be set before import pricing_engine)
MC_THREADS = 4
os.environ["OMP_NUM_THREADS"] = str(MC_THREADS)


REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent.parent
elif not (REPO / "CMakeLists.txt").exists():
    for p in REPO.parents:
        if (p / "CMakeLists.txt").exists():
            REPO = p
            break

sys.path.insert(0, str(REPO / "build-std"))

import pricing_engine as pe

MARKET_DATA_DIR = REPO / "data"
MARKET_JSON_BASE = MARKET_DATA_DIR / "market_snapshot.json"
OPTIONS_JSON = REPO / "data" / "options_book.json"
MAX_HORIZON_YEARS = 5.0
MC_SAMPLES = 100000
DYNAMICS = "lsv"
SEED = 41
RUN_VALIDATION = True

GENERATE_TEST_SNAPSHOTS = 0

In [2]:
def parse_iso(value: str) -> date:
    y, m, d = value.split("-")
    return date(int(y), int(m), int(d))


def format_iso(d: date) -> str:
    return f"{d.year:04d}-{d.month:02d}-{d.day:02d}"


def write_shifted_market_snapshots(base_path: Path, out_dir: Path, count: int) -> list[Path]:
    import QuantLib as ql

    base = json.loads(base_path.read_text())
    y, m, d = map(int, base["asof"].split("-"))
    cal = ql.TARGET()
    asof_ql = ql.Date(d, m, y)
    written: list[Path] = []
    for i in range(count):
        snap = dict(base)
        snap["asof"] = format_iso(date(asof_ql.year(), int(asof_ql.month()), asof_ql.dayOfMonth()))
        path = out_dir / f"market_snapshot_shift_{i:02d}.json"
        path.write_text(json.dumps(snap, indent=2) + "\n")
        written.append(path)
        if i + 1 < count:
            asof_ql = cal.advance(asof_ql, 1, ql.Days, ql.Following)
    return written


def simulation_horizon(specs: list[dict], ctx, max_years: float) -> str:
    latest = max(parse_iso(ctx.resolve_expiry_years(s["expiry_years"])) for s in specs)
    for s in specs:
        for obs in s.get("observation_dates") or []:
            if obs:
                latest = max(latest, parse_iso(obs))
    ay, am, ad = map(int, ctx.today.split("-"))
    cap = date(ay + int(max_years), am, min(ad, 28))
    return format_iso(min(latest, cap))


def count_ok(results) -> int:
    return len([r for r in results if r.status == "ok"])


if GENERATE_TEST_SNAPSHOTS:
    paths = write_shifted_market_snapshots(
        MARKET_JSON_BASE, MARKET_DATA_DIR, GENERATE_TEST_SNAPSHOTS
    )
    print(f"wrote {len(paths)} test snapshots:")
    for p in paths:
        print(f"  {p.name}")

specs = json.loads(OPTIONS_JSON.read_text())["options"]
market_files = sorted(MARKET_DATA_DIR.glob("market_snapshot*.json"))
if not market_files:
    raise FileNotFoundError(f"no market_snapshot*.json in {MARKET_DATA_DIR}")

print(f"options: {len(specs)}")
print(f"market files ({len(market_files)}):")
for p in market_files:
    asof = json.loads(p.read_text())["asof"]
    print(f"  {p.name}  asof={asof}")

options: 249
market files (1):
  market_snapshot.json  asof=2025-12-01


In [3]:
run_summaries = []

for market_path in market_files:
    market = json.loads(market_path.read_text())
    print(f"\n=== {market_path.name}  asof={market['asof']} ===")

    ctx = pe.PricingContext.from_tables(**market)
    ctx.preprocessing()
    ctx.calibration(run_validation=RUN_VALIDATION)

    sim_expiry = simulation_horizon(specs, ctx, MAX_HORIZON_YEARS)
    t0 = time.perf_counter()
    sim = ctx.simulate_paths(sim_expiry, mc_samples=MC_SAMPLES, dynamics=DYNAMICS, seed=SEED)
    sim_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    results = ctx.price_all(specs)
    price_ms = (time.perf_counter() - t0) * 1000

    n_ok = count_ok(results)
    n_err = len(results) - n_ok
    print(f"horizon={sim_expiry}  sim={sim_ms:.0f} ms  price_all={price_ms:.0f} ms  ok={n_ok}  err={n_err}")

    if n_err:
        for r in results:
            if r.status != "ok":
                print(f"  ERR {r.id}: {r.status[:120]}")
                break

    run_summaries.append({
        "market_file": market_path.name,
        "asof": market["asof"],
        "horizon": sim_expiry,
        "sim_ms": sim_ms,
        "price_ms": price_ms,
        "priced": n_ok,
        "errors": n_err,
        "results": results,
    })


=== market_snapshot.json  asof=2025-12-01 ===


TypeError: from_tables(): incompatible function arguments. The following argument types are supported:
    1. (asof: str, spot: float, rfr_tenor_years: list[float], rfr_zero_rates: list[float], repo_tenor_years: list[float], repo_zero_rates: list[float], vol_tenor_years: list[float], vol_strikes: list[float], implied_vols: list[float], dividend_dates: list[str] = [], dividend_amounts: list[float] = [], bergomi_k: float = 2.0, bergomi_nu: float = 1.0, bergomi_rho: float = -0.7) -> pricing_engine.PricingContext

Invoked with: kwargs: asof='2025-12-01', spot=8097.0, bergomi_k=2.0, bergomi_nu=1.0, bergomi_rho=-0.7, rfr_tenor_years=[0.0, 0.0055, 0.0192, 0.0833, 0.1667, 0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 20.0, 25.0, 30.0, 40.0, 50.0], rfr_zero_rates=[0.019301, 0.019301, 0.01929, 0.01961, 0.02011, 0.0206, 0.019248, 0.018877, 0.020715, 0.021105, 0.02198, 0.0229, 0.023745, 0.02453, 0.0253, 0.02604, 0.026725, 0.02735, 0.02795, 0.0285

In [ ]:
print("\n--- summary ---")
for s in run_summaries:
    print(
        f"{s['market_file']:32s}  asof={s['asof']}  horizon={s['horizon']}  "
        f"ok={s['priced']}  err={s['errors']}  sim={s['sim_ms']:.0f}ms  price={s['price_ms']:.0f}ms"
    )

# last market: quick price dump (optional)
last = run_summaries[-1]
priced_ok = [r for r in last["results"] if r.status == "ok"]
print(f"\nlast run ({last['market_file']}): {len(priced_ok)} prices")
for r in priced_ok:
    print(f"  {r.id}: {r.value:.4f}  (stderr={r.stderr:.4f})")
